# On reprend à partir du dataset coupé à 76000 images issu de jraigs_pipline_from_prepared.ipynb 

In [ ]:
from __future__ import annotations

import os
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
INPUT_CSV = PROJECT_ROOT / "datasets" / "jraigs_prepared" / "labels.csv"
SUBSETS_ROOT = PROJECT_ROOT / "datasets" / "jraigs_subsets"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "eda"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

csv_path = PROJECT_ROOT / "outputs" / "lum_color_blur_filtered.csv"

if not csv_path.exists():
    raise FileNotFoundError(f"Fichier introuvable: {csv_path}")

df_current = pd.read_csv(csv_path, low_memory=False)
print(f"Loaded: {csv_path}")
print(f"Rows={len(df_current)} | Cols={len(df_current.columns)}")
df_current.head(3)

df_current.describe()

: 

In [ ]:
base_df = df_current if "df_current" in globals() else df
image_col = "kept_image" if "kept_image" in base_df.columns else "source_image"

valid_images = base_df[
    base_df[image_col].notna()
    & base_df[image_col].astype(str).str.strip().ne("")
    & base_df[image_col].apply(lambda p: Path(str(p)).exists())
].copy()

if valid_images.empty:
    print("No valid image paths found.")
else:
    sample_n = min(10, len(valid_images))
    sample = valid_images.sample(n=sample_n)

    ncols = 5
    nrows = (sample_n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.5 * nrows))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for ax in axes:
        ax.axis("off")

    for i, (_, row) in enumerate(sample.iterrows()):
        img_path = Path(str(row[image_col]))
        try:
            img = plt.imread(img_path)
            axes[i].imshow(img)
            eye_id = row.get("Eye ID", "")
            label = row.get("Final Label", "")
            axes[i].set_title(f"{eye_id} | {label}", fontsize=9)
        except Exception:
            axes[i].text(0.5, 0.5, f"Load error\n{img_path.name}", ha="center", va="center", fontsize=8)
            axes[i].set_title("error", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)

: 